# Species migration portfolio assignment : Part 6

In this assignment I use data from the [GBIF]() on the barn swallow migration 

## Set up

In [ ]:
### Bring in stored var 
%store -r

### Checek for ecoreg_gdf, portfolio_gbif_gdf, portfolio_gbif_path, gbif_dir

In [ ]:
### import packages

### libraries for dynamic mapping

### manage coordinate ref systems
import cartopy.crs as ccrs

### customize dynamic plot
import panel as pn

### make dynamic plot with geodataframe
import hvplot.pandas
import holoviews as hv

### Run this fix if the plot is not showing, once per kernal session
hvplot.extension("bokeh")

### get month names
import calendar

### folder/file management
import os
import pathlib

### tabular data
import pandas as pd

### geospatial vactor data
import geopandas as gpd
import earthpy as et

## Step 2: Load (GBIF) geodataframe


## Step 3: Connect the GBIF observations to the ecoregions

### Step 3A: Identify the ecoregion for each observation

Here I combine the ecoregions and the observations using a spatial join.

In [ ]:
### join the gbif observations to the ecoregion polygons
portfolio_gbif_ecoregion_gdf = (
    ecoreg_gdf
    
    ### match the CRS of the gbif data and the ecoregions
    .to_crs(portfolio_gbif_gdf.crs)

    ### find ecoregion for each observation
    .sjoin(
        portfolio_gbif_gdf,
        ### type of database join
        how = 'inner', 
        ###
        predicate = 'contains')

    ### any additional manipulations you want to do )
    .rename(columns = {'OBJECTID': 'ecoregion'})
)
### check it out
portfolio_gbif_ecoregion_gdf

### Step 3B: Count the observations in each ecoregion each month

First, group the GBIF observations by ecoregion:
1. Count the observations in each ecoregion by month.
2. Select only month/ecoregion combinations that have more than one occurrence recorded
3. Compute the mean occurrences by ecoregion and by month.
4. Normalize the number of occurrences by month and ecoregion.

In [ ]:
### summarize the occurrences by ecoregion/month combination
portfolio_occurrence_df = (
    portfolio_gbif_ecoregion_gdf

    ### group by each ecoregion and month 
    .groupby(['ecoregion', 'month'])

    ### and count the number of occurrences
    .agg(occurrences=('ecoregion', 'count'))
)
portfolio_occurrence_df.head()

### get rid of rare observations (which are possibly misidentifications)
portfolio_occurrence_df = portfolio_occurrence_df[portfolio_occurrence_df.occurrences > 1]

### calculate the mean # observations by ecoregion
portfolio_mean_occurrences_by_ecoregion = (
    portfolio_occurrence_df
    .groupby('ecoregion')
    .mean()
)

### calculate the mean # observations by month
portfolio_mean_occurrences_by_month = (
    portfolio_occurrence_df
    .groupby('month')
    .mean()
)

portfolio_mean_occurrences_by_ecoregion, portfolio_mean_occurrences_by_month

### Step 3C: Normalize the observations

Divide occurrences by the mean occurrences by month and by ecoregion.

In [ ]:
### normalize by space and time for sampling effort
portfolio_occurrence_df['norm_occurrences'] = (
    portfolio_occurrence_df
    / portfolio_mean_occurrences_by_ecoregion
    / portfolio_mean_occurrences_by_month
)

### check it out
portfolio_occurrence_df